# ***1. Objetivo***

Esta atividade teve como objetivo aplicar boas práticas de segurança no Google Colab utilizando Secrets, além de realizar a extração de informações de um documento PDF da Relação Municipal de Medicamentos Essenciais (REMUME) de Criciúma/SC. O trabalho demonstra como acessar dados não estruturados e transformá-los em informações utilizáveis para análise.


# 2. Segurança em Primeiro Lugar – Colab Secrets ***negrito***

Para evitar a exposição de credenciais no código-fonte, foi utilizado o sistema de Secrets do Google Colab.


In [ ]:
from google.colab import userdata

token = userdata.get('TOKEN_API')

print("Token carregado com sucesso!")

Token carregado com sucesso!


# 3. Extração de Dados do PDF REMUME

O documento utilizado foi a Relação Municipal de Medicamentos Essenciais (REMUME) do município de Criciúma, versão novembro de 2024.

In [ ]:
!pip install pdfplumber

**Leitura do PDF**

In [ ]:
import pdfplumber

arquivo_pdf = "remume_versao_2024.pdf"

with pdfplumber.open(arquivo_pdf) as pdf:
    print(f"Total de páginas: {len(pdf.pages)}")

    for i, pagina in enumerate(pdf.pages):
        texto = pagina.extract_text()

        if texto:
            print(f"\n--- Página {i+1} ---\n")
            print(texto[:1000])  # mostra os primeiros 1000 caracteres

Total de páginas: 28

--- Página 1 ---

GOVERNO DE CRICIÚMA
PODER EXECUTIVO
SECRETARIA MUNICIPAL DE SAÚDE
RELAÇÃO MUNICIPAL DE MEDICAMENTOS ESSENCIAIS (REMUME)
LISTA DE MEDICAMENTOS DISPONIBILIZADOS
PELA REDE MUNICIPAL DE SAÚDE DE CRICIÚMA
Versão de atualização: novembro/2024

--- Página 2 ---

Prefeitura Municipal de Criciúma - Secretaria de Saúde - Assistência Farmacêutica - Relação Municipal de
Medicamentos - REMUME
REMUME CRICIÚMA - Versão 11/2024 - Página 2 de 27

--- Página 3 ---

Prefeitura Municipal de Criciúma - Secretaria de Saúde - Assistência Farmacêutica - Relação Municipal de
Medicamentos - REMUME
Equipe Técnica
Heby Varnier Mariano
Farmacêutica Coordenadora Assistência Farmacêutica
Larissa de Oliveira de Batista
Farmacêutica Coordenadora Assistência Farmacêutica
Comissão de Farmácia e Terapêutica - CFT
Bruna Karoline Arruda da Silva
Enfermeira da Atenção Básica
Monica Dalmolin Soratto Daniel
Enfermeira da Atenção Básica
Paulo Henrique Fabiano Zamora
Médico da Atenção Bás

**Limpeza e Tratamento dos Dados**

In [ ]:
import re
import pdfplumber

# 'arquivo_pdf' is available from a previous cell, but re-defining for self-containment/clarity
arquivo_pdf = "remume_versao_2024.pdf"

texto_completo = ""
with pdfplumber.open(arquivo_pdf) as pdf:
    for pagina in pdf.pages:
        extracted_text = pagina.extract_text()
        if extracted_text:
            texto_completo += extracted_text + "\n"

texto_limpo = texto_completo

# Remove múltiplos espaços
texto_limpo = re.sub(r'\s+', ' ', texto_limpo)

# Remove quebras de linha excessivas
texto_limpo = texto_limpo.strip()

print(texto_limpo[:1000])

GOVERNO DE CRICIÚMA PODER EXECUTIVO SECRETARIA MUNICIPAL DE SAÚDE RELAÇÃO MUNICIPAL DE MEDICAMENTOS ESSENCIAIS (REMUME) LISTA DE MEDICAMENTOS DISPONIBILIZADOS PELA REDE MUNICIPAL DE SAÚDE DE CRICIÚMA Versão de atualização: novembro/2024 Prefeitura Municipal de Criciúma - Secretaria de Saúde - Assistência Farmacêutica - Relação Municipal de Medicamentos - REMUME REMUME CRICIÚMA - Versão 11/2024 - Página 2 de 27 Prefeitura Municipal de Criciúma - Secretaria de Saúde - Assistência Farmacêutica - Relação Municipal de Medicamentos - REMUME Equipe Técnica Heby Varnier Mariano Farmacêutica Coordenadora Assistência Farmacêutica Larissa de Oliveira de Batista Farmacêutica Coordenadora Assistência Farmacêutica Comissão de Farmácia e Terapêutica - CFT Bruna Karoline Arruda da Silva Enfermeira da Atenção Básica Monica Dalmolin Soratto Daniel Enfermeira da Atenção Básica Paulo Henrique Fabiano Zamora Médico da Atenção Básica da Rede Municipal de Saúde Semina Nakos Cirurgiã-Dentista da Atenção Básic

***Extração das Tabelas***

In [ ]:
import pdfplumber

with pdfplumber.open("remume_versao_2024.pdf") as pdf:

    for i, pagina in enumerate(pdf.pages):

        tabelas = pagina.extract_tables()

        for tabela in tabelas:

            print(f"\nTabela encontrada na página {i+1}")

            for linha in tabela:
                print(linha)


Tabela encontrada na página 6
['Item', 'Medicamento', 'Apresentação', 'Local de acesso', None, 'Classificação e tipo de\nreceita']
['01', 'Acetilcisteína', '600 mg\ngranulado\nenvelopes com\n5 g', '- Farmácias Distritais', None, '- Expectorante\n- Receituário simples']
['02', 'Aciclovir', '200 mg\nComprimido', '- Farmácias Distritais\n- Farmácia Estratégica', None, '- Antiviral\n- Receituário simples']
['03', 'Aciclovir', '50 mg/g\nCreme bisnaga', '', '- Farmácias Distritais', '- Antiviral\n- Receituário simples']
[None, None, None, None, '- Farmácia Estratégica', None]
[None, None, None, None, '- Dispensários nas UBS', None]
['04', 'Ácido acetilsalicílico', '100 mg\nComprimido', '- Farmácias Distritais\n- Dispensários nas UBS', None, '- Antitrombótico\n- Receituário simples']
['05', 'Ácido fólico', '5 mg\nComprimido', '- Farmácias Distritais\n- Farmácia Estratégica\n- Dispensários nas UBS', None, '- Vitamina\n- Receituário simples']
['06', 'Ácido fólico', '0,2 mg/ml\nSolução oral\nFr

***Armazenamento dos Dados em CSV***

In [ ]:
import pandas as pd
import pdfplumber

dados = []

with pdfplumber.open("remume_versao_2024.pdf") as pdf:

    for pagina in pdf.pages:

        tabelas = pagina.extract_tables()

        for tabela in tabelas:

            for linha in tabela:
                dados.append(linha)

df = pd.DataFrame(dados)

df.to_csv(
    "medicamentos_remume.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo CSV criado com sucesso!")

Arquivo CSV criado com sucesso!


Foi utilizada a biblioteca pdfplumber para realizar a leitura e extração das informações presentes no documento REMUME 2024 do município de Criciúma. O script percorreu todas as páginas do PDF, obtendo o conteúdo textual de forma automatizada. Os dados extraídos incluem informações sobre medicamentos, apresentações farmacêuticas, locais de acesso e classificação terapêutica, demonstrando a capacidade de manipular documentos PDF não estruturados em Python

